# Masking Model — Registration

Logs and registers `MaskingModel` in Unity Catalog with MLflow.

## Prerequisites
- Databricks Runtime ML 14.x or later
- Unity Catalog enabled
- Model code synced to workspace under `../model/`

## Next Steps
1. Run `deploy_endpoint.py` with `MODEL_NAME = "doc_masking"` and the version printed below
2. Set `DATABRICKS_MASKING_ENDPOINT` on the FastAPI backend

In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
# ---------------------------------------------------------------------------
# Configuration — update before running
# ---------------------------------------------------------------------------
CATALOG    = "datascience_dev_bronze_sandbox"
SCHEMA     = "ds_document_deidentification"
MODEL_NAME = "privasee_doc_masking"

UC_MODEL_PATH  = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
UC_VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/sessions"

print(f"Model : {UC_MODEL_PATH}")
print(f"Volume: {UC_VOLUME_PATH}")

In [0]:
import json
import os
import sys

import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import ColSpec, Schema

# Add repo root to path so relative imports inside model/ work
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
module_dir = os.path.join("/Workspace" + os.path.dirname(notebook_path), "../../..")
sys.path.insert(0, module_dir)
print(f"Module dir: {module_dir}")

In [0]:
from privasee.databricks.model.masking_model import MaskingModel
print("MaskingModel imported successfully")

In [0]:
# ---------------------------------------------------------------------------
# MLflow signature
# ---------------------------------------------------------------------------
input_schema = Schema([
    ColSpec("string",  "session_id"),
    ColSpec("string",  "entities_to_mask"),   # JSON string: list of entity dicts
    ColSpec("boolean", "run_verification", required=False),  # default False
])

output_schema = Schema([
    ColSpec("string", "session_id"),
    ColSpec("string", "status"),
    ColSpec("long",   "entities_masked"),
    ColSpec("long",   "occurrences_total",   required=False),
    ColSpec("long",   "occurrences_masked",  required=False),
    ColSpec("double", "score",               required=False),
    ColSpec("string", "error_message",       required=False),
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)
print("Signature defined")

In [0]:
# ---------------------------------------------------------------------------
# Input example
# ---------------------------------------------------------------------------
input_example = pd.DataFrame({
    "session_id": ["example_session_123"],
    "entities_to_mask": [json.dumps([
        # {
        #     "id": "e1",
        #     "entity_type": "Full Name",
        #     "original_text": "John Smith",
        #     "replacement_text": "Jane Doe",
        #     "bounding_box": [0.05, 0.08, 0.2, 0.025],
        #     "bounding_boxes": [
        #         {
        #             "x": 0.14965359893857966,
        #             "y": 0.16382730180575378,
        #             "width": 0.11323988798943746,
        #             "height": 0.015998384227133816
        #         }
        #     ],
        #     "strategy": "Fake Data",
        #     "approved": True,
        #     "page_number": 1,
        # }
        {
            "id": "e1",
            "entity_type": "Full Name",
            "original_text": "John Smith",
            "replacement_text": "Jane Doe",
            "strategy": "Fake Data",
            "approved": True,
            "occurences": [
                {
                    "page_number": 1,
                    "bounding_box": [0.05, 0.08, 0.2, 0.025],
                    "original_text": "John Smith",
                }
            ]
        }    ])],
})
print("Input example created")

In [0]:
# ---------------------------------------------------------------------------
# Log model to MLflow
# ---------------------------------------------------------------------------
mlflow.set_registry_uri("databricks-uc")

# username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
# mlflow.set_experiment(f"/Users/{username}/privasee_experiments")
experiment_path = "/datascience/projects/ds_document_deidentification_tool/privasee"
mlflow.set_experiment(experiment_path)

requirements_path = os.path.join(os.path.dirname(module_dir), "databricks/requirements.txt")

with mlflow.start_run(run_name="privasee_masking_model_registration") as run:
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=MaskingModel(),
        signature=signature,
        input_example=input_example,
        pip_requirements=requirements_path,
        code_paths=[os.path.join(module_dir, "privasee")],
    )
    mlflow.log_param("catalog", CATALOG)
    mlflow.log_param("schema", SCHEMA)
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("uc_volume_path", UC_VOLUME_PATH)

print(f"Model logged: {model_info.model_uri}")

In [0]:
# ---------------------------------------------------------------------------
# Register in Unity Catalog
# ---------------------------------------------------------------------------
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=UC_MODEL_PATH,
    tags={"project": "privasee", "type": "masking", "AppSpace":"A-007100"},
)

MODEL_VERSION = str(registered_model.version)
print(f"Registered: {UC_MODEL_PATH} v{MODEL_VERSION}")

In [0]:
print(f"""
Model Path:    {UC_MODEL_PATH}
Model Version: {MODEL_VERSION}

Next: run deploy_endpoint.py with:
  MODEL_NAME    = \"{MODEL_NAME}\"
  MODEL_VERSION = \"{MODEL_VERSION}\"

Then set on FastAPI backend:
  DATABRICKS_MASKING_ENDPOINT = https://<host>/serving-endpoints/<endpoint-name>/invocations
""")

## Test Masking Model Locally

Before deploying to an endpoint, test the model works correctly with sample data.

In [0]:
# ---------------------------------------------------------------------------
# Test masking model locally before deploying
# ---------------------------------------------------------------------------
import mlflow
import pandas as pd
import json
import os

# Set environment variables for model (normally set in endpoint config)
os.environ['UC_VOLUME_PATH'] = UC_VOLUME_PATH
os.environ['DATABRICKS_HOST'] = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"
os.environ['DATABRICKS_TOKEN'] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Load the logged model
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

# Use the integration_test_002 session (already has original.pdf in UC volume)
# Note: Vision API returned 0 entities in the earlier run, so we'll use sample entities for testing
test_input = pd.DataFrame({
    "session_id": ["integration_test_002"],
    "entities_to_mask": [json.dumps([
                {
                    "entity_type": "Claimant Name",
                    "original_text": "Stephen Parrot",
                    "confidence": 0.99,
                    "id": "76673463-773e-4100-bf63-7d31f1b3c72d",
                    "approved": True,
                    "strategy": "Fake Data",
                    "replacement_text": "William Long",
                    "occurrences": [
                        {
                            "page_number": 1,
                            "bounding_boxes": [[
                            0.14965359893857966,
                            0.16382730180575378,
                            0.11323988798943746,
                            0.015998384227133816
                            ]],
                            "original_text": "Stephen Parrot"
                        },
                        {
                            "page_number": 1,
                            "bounding_boxes": [[
                            0.12094335670594705,
                            0.20772694728975996,
                            0.062211488372183626,
                            0.015998384227133816
                            ]],
                            "original_text": "Stephen"
                        }
                    ]
                },
                {
                    "entity_type": "incident_date",
                    "original_text": "12 January 2020",
                    "confidence": 0.95,
                    "replacement_text": "incident_date_A",
                    "occurrences": [
                        {
                            "page_number": 1,
                            "original_text": "12 January 2020,",
                            "bounding_boxes": [
                                [
                                0.6438296945689354,
                                0.286119207615939,
                                0.12703841394716664,
                                0.015998384227133844
                                ]
                            ]
                        }
                    ],
                    "id": "8418c928-1aff-4372-958c-be9461b408c1",
                    "approved": True,
                    "strategy": "Entity Label"
                }
            ])
    ],
})

print("Test Input:")
print(f"  Session ID: {test_input['session_id'][0]}")
print(f"  Entities: {len(json.loads(test_input['entities_to_mask'][0]))}")
print(f"  Document: /Volumes/{CATALOG}/{SCHEMA}/sessions/integration_test_002/original.pdf")
print("\nRunning prediction...")

# Run prediction
try:
    result = loaded_model.predict(test_input)
    print("\n✓ Prediction successful!")
    print("\nResult:")
    display(result)
    
    # Check output
    if result['status'][0] == 'complete':
        print(f"\n✓ Status: {result['status'][0]}")
        print(f"✓ Entities masked: {result['entities_masked'][0]}")
        print(f"\n📄 Masked PDF written to:")
        print(f"   /Volumes/{CATALOG}/{SCHEMA}/sessions/integration_test_002/masked.pdf")
        print(f"\n🎉 Model is ready to deploy!")
    else:
        print(f"\n⚠️  Status: {result['status'][0]}")
        if 'error_message' in result.columns:
            print(f"Error: {result['error_message'][0]}")
except Exception as e:
    print(f"\n❌ Prediction failed: {str(e)}")
    import traceback
    traceback.print_exc()
    raise